### Import packages

In [2]:
import os
from openai import OpenAI
import pandas as pd
import json
from tqdm.auto import tqdm

### Check if environment variable is set for OpenAI API Key

In [3]:
try:
    api_key = os.environ["OPENAI_API_KEY"]
except KeyError as exc:
    raise RuntimeError("Environment variable OPENAI_API_KEY missing.") from exc

### Define which OpenAI model to use

In [4]:
MODEL = 'gpt-5-nano'

### Construct OpenAI client instance

In [5]:
client = OpenAI(api_key=api_key)

def send_LLM(prompt: str) -> str:
    try:
        # Send chat request
        chat_completion = client.chat.completions.create(
            model = MODEL,
            messages=[
                {"role": "system", "content": "You are a rigorous JSON-only annotator of online irony."},
                {"role": "user", "content": prompt}
            ])
    except KeyError as exc:
        raise RuntimeError("Error") from exc

    return chat_completion.choices[0].message.content

### Define schema for irony as JSON

In [6]:
IRONY_SCHEMA = {
    "irony": {
        "irony_score": "1-5 integer (1 = not ironic, 5 = very ironic)",
        "confidence_score": "1-5 integer (1 = not confident, 5 = very confident)",
        "justification": "string - explanation based on observed language",
        "irony_markers": ["string"]
    } 
}

def prompt_irony_analysis(thread_title: str, comment: str) -> str:
    schema_str = json.dumps(IRONY_SCHEMA, indent=2)
    return f"""
You are a rigorous JSON-only annotator of online irony.

Your task is to analyze whether a given Reddit comment contains irony.
The comments come from the “r/de” subreddit.
The goal is to annotate comments based on the definition and use the results to provide a foundation for a scientific analysis. 

Definition:  

Irony is a rhetorical device in which a statement deliberately creates a discrepancy between its literal meaning and its intended meaning.  

Analyze the COMMENT under the THREAD TITLE below.

Return ONLY one JSON object that strictly matches this schema:

{schema_str}

THREAD TITLE:
\"\"\"{thread_title}\"\"\"

COMMENT:
\"\"\"{comment}\"\"\"

"""

### Read input data

In [7]:
df = pd.read_csv('dataset.csv')

df.head()

,collected_by,id,thread_title,comment
0,Johan,1,Mehrheit sieht ältere im Vorteil – nicht einma...,Solange alle ihre Tiefkühlpizza und RTL haben ...
1,Johan,2,Mehrheit sieht ältere im Vorteil – nicht einma...,Ich bin schockiert.
2,Johan,3,Mehrheit sieht ältere im Vorteil – nicht einma...,"Würden sich die babyboomer zu Tode arbeiten, b..."
3,Johan,4,Mehrheit sieht ältere im Vorteil – nicht einma...,Auf den Schreck erstmal eine Rentenerhöhung.
4,Johan,5,Mehrheit sieht ältere im Vorteil – nicht einma...,Finde den Generationen-Vertrag an sich schon d...


In [8]:
df.shape

(60, 4)

### Perform LLM irony analysis on every comment
We had 9 human annotators and as LLMs are non-deterministic, we also perform it 9 times to see potential variability.

In [9]:
os.makedirs("llm_raw", exist_ok=True)

In [10]:


results = []
num_of_iterations = 9

for iteration in tqdm(range(2, num_of_iterations + 1), desc="LLM iterations"):

    iteration_results = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"Processing comments (iter {iteration})",
        leave=False
    ):
        row_id = row["id"]
        thread_title = row["thread_title"]
        comment = row["comment"]

        prompt = prompt_irony_analysis(thread_title, comment)
        llm_answer = send_LLM(prompt)

        # Parse JSON safely
        try:
            parsed = json.loads(llm_answer)
            irony_obj = parsed.get("irony", {})
        except json.JSONDecodeError:
            irony_obj = {
                "error": "invalid_json",
                "raw_response": llm_answer
            }

        iteration_results.append({
            "id": row_id,
            "thread_title": thread_title,
            "comment": comment,
            "irony": irony_obj
        })

    # Save one file per iteration
    output_path = f"llm_raw/iteration_{iteration:02d}.json"

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(iteration_results, f, ensure_ascii=False, indent=2)

    print(f"Saved: {output_path}")

LLM iterations:   0%|          | 0/8 [00:00<?, ?it/s]

Processing comments (iter 2):   0%|          | 0/60 [00:00<?, ?it/s]

Saved: llm_raw/iteration_02.json


Processing comments (iter 3):   0%|          | 0/60 [00:00<?, ?it/s]

Saved: llm_raw/iteration_03.json


...

Saved: llm_raw/iteration_07.json


Processing comments (iter 8):   0%|          | 0/60 [00:00<?, ?it/s]

Saved: llm_raw/iteration_08.json


Processing comments (iter 9):   0%|          | 0/60 [00:00<?, ?it/s]

Saved: llm_raw/iteration_09.json
